In [1]:
import os

In [2]:
%pwd

'c:\\Projects\\Kidney-Disease-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Projects\\Kidney-Disease-Classification'

In [13]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen = True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [14]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [15]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir = Path(config.root_dir),
            base_model_path = Path(config.base_model_path),
            updated_base_model_path = Path(config.updated_base_model_path),
            params_image_size = self.params.IMAGE_SIZE,
            params_learning_rate = self.params.LEARNING_RATE,
            params_include_top = self.params.INCLUDE_TOP,
            params_weights = self.params.WEIGHTS,
            params_classes = self.params.CLASSES
        )
        return prepare_base_model_config 

In [16]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [21]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

In [22]:
class PrepareBaseModel:

    def __init__(self, config):
        self.config = config

    def get_base_model(self):

        self.model = tf.keras.applications.MobileNetV2(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(
        model,
        classes,
        freeze_all,
        freeze_till,
        learning_rate
    ):

        # ---------------- Freeze Layers ----------------

        if freeze_all:
            model.trainable = False

        elif freeze_till is not None and freeze_till > 0:

            # Make all layers trainable first
            model.trainable = True

            # Freeze all except last 'freeze_till' layers
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        # ---------------- Classification Head ----------------

        x = model.output

        x = tf.keras.layers.GlobalAveragePooling2D()(x)

        x = tf.keras.layers.Dense(32,activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)

        x = tf.keras.layers.Dropout(0.7)(x)

        outputs = tf.keras.layers.Dense(
            units=classes,
            activation="softmax",
            dtype="float32"
        )(x)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=outputs
        )

        # ---------------- Compile ----------------

        full_model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=["accuracy"]
        )

        full_model.summary()

        return full_model

    def update_base_model(self):

        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=False,
            freeze_till=20,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [19]:
try: 
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config = prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-07-18 14:45:02,242: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-18 14:45:02,247: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-18 14:45:02,247: INFO: common: created directory at: artifacts]
[2026-07-18 14:45:04,068: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 